In [3]:
import json
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter


# =========================
# 1. PDF EXTRACTION
# =========================
def extract_text_with_fitz(pdf_path: str) -> str:
    """Extract text from PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pages.append(page.get_text("text"))
    return "\n".join(pages)


# =========================
# 2. CLEANING
# =========================
def clean_legal_pdf_text(text: str) -> str:
    """
    Clean common legal PDF artifacts:
    - Công báo headers/footers
    - isolated page numbers
    - excessive spaces/newlines
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove Công báo header/footer
    text = re.sub(
        r"CÔNG BÁO/Số\s+\d+\s*\+\s*\d+/Ngày\s+\d{1,2}-\d{1,2}-\d{4}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove isolated page numbers
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)

    # Remove repeated spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Compress excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Xóa footnote số cuối trang kiểu: "2 Cụm từ..."
    text = re.sub(
        r"\n\s*\d{1,2}\s+Cụm từ.+?(?=\n[A-ZĐÀÁẠẢÃÂẦẤẬẨẪ]|\Z)",
        "",
        text,
        flags=re.DOTALL
    )

    return text.strip()


# =========================
# 3. PARSING CHAPTERS + ARTICLES
# =========================
CHAPTER_REGEX = re.compile(
    r"(Chương\s+[IVXLCDM]+)\s*\n+\s*([^\n]+(?:\n(?!Điều\s|\nChương\s)[^\n]+)*)",
    flags=re.UNICODE
)

ARTICLE_REGEX = re.compile(
    r"(Điều\s+\d+[a-z]?\.\s*[^\n]+)",
    flags=re.UNICODE
)


def parse_chapters(text: str) -> List[Dict]:
    """
    Return chapter blocks:
    [
      {
        'chapter_label': 'Chương I',
        'chapter_title': 'NHỮNG QUY ĐỊNH CHUNG',
        'chapter_full': 'Chương I NHỮNG QUY ĐỊNH CHUNG',
        'content': '...'
      }
    ]
    """
    matches = list(CHAPTER_REGEX.finditer(text))
    chapters = []

    if not matches:
        return [{
            "chapter_label": None,
            "chapter_title": None,
            "chapter_full": None,
            "content": text
        }]

    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        chapter_label = match.group(1).strip()
        chapter_title = match.group(2).strip()
        chapter_full = f"{chapter_label} {chapter_title}"

        content_start = match.end()
        content = text[content_start:end].strip()

        chapters.append({
            "chapter_label": chapter_label,
            "chapter_title": chapter_title,
            "chapter_full": chapter_full,
            "content": content
        })

    return chapters


def split_article_blocks(chapter_content: str) -> List[Dict]:
    """
    Split a chapter content into article blocks using finditer.
    Returns:
    [
      {'article_title': 'Điều 1. ...', 'article_content': '...'}
    ]
    """
    matches = list(ARTICLE_REGEX.finditer(chapter_content))
    articles = []

    if not matches:
        return []

    for i, match in enumerate(matches):
        article_title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(chapter_content)
        article_content = chapter_content[start:end].strip()

        articles.append({
            "article_title": article_title,
            "article_content": article_content
        })

    return articles


# =========================
# 4. SPLIT LONG ARTICLES
# =========================
def split_by_clause(article_text: str) -> List[str]:
    """
    Try splitting legal text by numbered clauses:
    1. ...
    2. ...
    3. ...
    """
    clause_pattern = re.compile(r"(?=\n?\s*\d+\.\s)")
    parts = clause_pattern.split(article_text)

    cleaned = [p.strip() for p in parts if p.strip()]
    return cleaned if len(cleaned) > 1 else [article_text.strip()]


def split_long_context(title: str, context: str, max_length: int = 1000) -> List[Dict]:
    """
    Split long contexts:
    1) Prefer splitting by legal clauses
    2) If still too long, use RecursiveCharacterTextSplitter
    """
    clause_parts = split_by_clause(context)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_length,
        chunk_overlap=150,
        separators=["\n\n", "\n", "; ", ". ", " ", ""]
    )

    chunks = []
    for part_idx, part in enumerate(clause_parts, start=1):
        if len(part) <= max_length:
            chunks.append({
                "title": title,
                "context": part.strip(),
                "sub_part": part_idx
            })
        else:
            sub_chunks = splitter.split_text(part)
            for sub_idx, sub_chunk in enumerate(sub_chunks, start=1):
                chunks.append({
                    "title": title,
                    "context": sub_chunk.strip(),
                    "sub_part": f"{part_idx}.{sub_idx}"
                })

    return chunks


# =========================
# 5. FULL PROCESSING
# =========================
def process_legal_pdf(
    pdf_path: str,
    source_file: str,
    law_name: Optional[str] = None,
    domain: Optional[str] = None,
    effective_date: Optional[str] = None,
    law_id: Optional[str] = None,
    max_context_len: int = 1000
) -> List[Dict]:
    raw_text = extract_text_with_fitz(pdf_path)
    cleaned_text = clean_legal_pdf_text(raw_text)

    chapters = parse_chapters(cleaned_text)
    final_chunks = []

    for chapter in chapters:
        chapter_full = chapter["chapter_full"]
        articles = split_article_blocks(chapter["content"])

        for article in articles:
            article_title = article["article_title"]
            article_content = article["article_content"].strip()

            if not article_content:
                continue

            combined_title = (
                f"{article_title} | {chapter_full}"
                if chapter_full else article_title
            )

            base_payload = {
                "law_name": law_name,
                "source_file": source_file,
                "domain": domain,
                "effective_date": effective_date,
                "law_id": law_id,
                "chapter": chapter_full,
                "article": article_title,
                "title": combined_title
            }

            if len(article_content) <= max_context_len:
                final_chunks.append({
                    **base_payload,
                    "context": article_content
                })
            else:
                sub_chunks = split_long_context(
                    title=combined_title,
                    context=article_content,
                    max_length=max_context_len
                )
                for sc in sub_chunks:
                    final_chunks.append({
                        **base_payload,
                        "context": sc["context"],
                        "sub_part": sc.get("sub_part")
                    })

    return final_chunks


# =========================
# 6. SAVE OUTPUT
# =========================
def save_to_json(data: List[Dict], output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# =========================
# 7. MAIN
# =========================
if __name__ == "__main__":
    output_json = "output_qdrant_ready.json"

    chunks = process_legal_pdf(
        pdf_path="E:/VN-legal-chatbot/data_processing/2026_175_47_VBHN-VPQH.pdf",
        source_file="Luat_BVQLNTD.pdf",
        law_name="Luật Bảo vệ quyền lợi người tiêu dùng 2023",
        domain="tieu_dung",
        effective_date="2024-07-01",
        law_id="BVQLNTD_2023"
    )

    print(f"Total chunks created: {len(chunks)}")
    if chunks:
        print("Sample chunk:")
        print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

    save_to_json(chunks, output_json)
    print(f"Saved to {output_json}")

Total chunks created: 284
Sample chunk:
{
  "law_name": "Luật Bảo vệ quyền lợi người tiêu dùng 2023",
  "source_file": "Luat_BVQLNTD.pdf",
  "domain": "tieu_dung",
  "effective_date": "2024-07-01",
  "law_id": "BVQLNTD_2023",
  "chapter": "Chương I NHỮNG QUY ĐỊNH CHUNG",
  "article": "Điều 1. Phạm vi điều chỉnh",
  "title": "Điều 1. Phạm vi điều chỉnh | Chương I NHỮNG QUY ĐỊNH CHUNG",
  "context": "Luật này quy định về nguyên tắc, chính sách bảo vệ quyền lợi người tiêu \ndùng; quyền và nghĩa vụ của người tiêu dùng; trách nhiệm của tổ chức, cá nhân \nkinh doanh đối với người tiêu dùng; hoạt động bảo vệ quyền lợi người tiêu dùng \ncủa cơ quan, tổ chức; giải quyết tranh chấp giữa người tiêu dùng và tổ chức, cá \nnhân kinh doanh; quản lý nhà nước về bảo vệ quyền lợi người tiêu dùng."
}
Saved to output_qdrant_ready.json
